In [ ]:
import pandas as pd
import json

In [ ]:
all_schedules = []

with open('./data/timetable.jsonl', 'r') as file:
    for line in file:
        json_record = json.loads(line)
        if 'timetable' in json_record:
            timetable = json_record['timetable']  
            if 'routes' in timetable:
                routes = timetable['routes'] 
                for route in routes:
                    if 'schedules' in route:
                        schedules = route['schedules']
                        all_schedules.extend(schedules)

len(all_schedules)

In [ ]:
schedules_df = pd.DataFrame(all_schedules)
display(schedules_df.head())

In [ ]:
schedules_df.drop(['$type','periods'], axis=1, inplace=True)

def clean_journey(journey_list):
    if isinstance(journey_list, list):
        return ["{}:{}".format(journey['hour'], journey['minute']) for journey in journey_list]
    elif isinstance(journey_list, dict):
        return "{}:{}".format(journey_list['hour'], journey_list['minute'])
    return journey_list

schedules_df['knownJourneys'] = schedules_df['knownJourneys'].apply(clean_journey)
schedules_df['firstJourney'] = schedules_df['firstJourney'].apply(clean_journey)
schedules_df['lastJourney'] = schedules_df['lastJourney'].apply(clean_journey)

schedules_df['knownJourneys'] = schedules_df['knownJourneys'].apply(lambda x: ', '.join(x))
schedules_df

In [ ]:
line_and_stationId= []

with open('./data/timetable.jsonl', 'r') as file:
    for line in file:

        json_line = json.loads(line)
        line_name = json_line.get("lineName", None)
        station_ids = [station.get("stationId", None) for station in json_line.get("stations", []) if "stationId" in station]
        
        line_and_stationId.append({
            "LineName": line_name,
            "StationIds": station_ids
        })

print(len(line_and_stationId))
line_and_stationId_df = pd.DataFrame(line_and_stationId)
line_and_stationId_df.head()

In [ ]:
line_and_stationId_df.explode('StationIds')

In [ ]:
station_ids_to_keep = {"490G00000350", "490G00003195", "490G00019703"}

line_and_stationId_filtered = [
    {
        "LineName": entry["LineName"],
        "StationIds": [sid for sid in entry["StationIds"] if sid in station_ids_to_keep]
    }
    for entry in line_and_stationId
]

line_and_stationId_filtered_df = pd.DataFrame(line_and_stationId_filtered)
line_and_stationId_filtered_df.explode("StationIds")

In [ ]:
all_intervals = []

with open('./data/timetable.jsonl', 'r') as file:
    for line in file:

        json_record = json.loads(line)
        if 'timetable' in json_record:
            timetable = json_record['timetable']  
            if 'routes' in timetable:
                routes = timetable['routes'] 
                for route in routes:
                    if 'stationIntervals' in route:
                        station_intervals = route['stationIntervals'] 
                        for station_interval in station_intervals:
                            if 'intervals' in station_interval:
                                intervals = station_interval['intervals']
                                all_intervals.extend(intervals)

len(all_intervals), all_intervals[:2]

In [ ]:
intervals_df = pd.DataFrame(all_intervals)

print("Before filtering:", intervals_df.shape[0])
selected_stop_ids = ['490000112M', '490003191F', '490019703Z']

filtered_intervals_df = intervals_df[intervals_df['stopId'].isin(selected_stop_ids)]
print("After filtering:", filtered_intervals_df.shape[0])

filtered_intervals_df = filtered_intervals_df.copy()
filtered_intervals_df.drop('$type', axis=1, inplace=True)
filtered_intervals_df.head()

In [ ]:
temp_dfs = []  
with open('./data/arrival.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)  
        temp_df = pd.json_normalize(data) 
        temp_dfs.append(temp_df) 

arrivals_df = pd.concat(temp_dfs)    
arrivals_df.head()

In [ ]:
columns_to_drop = ['$type','naptanId','id', 'operationType', 'bearing', 'vehicleId', 'lineId', 'destinationNaptanId', 
                   'timing.$stype', 'timing.$type', 'timing.insert', 'timing.read', 'currentLocation', 'timeToStation',
                  'timing.sent', 'platformName', 'timing.received', 'modeName', 'timing.countdownServerAdjustment']
arrivals_df = arrivals_df.drop(columns=columns_to_drop, axis=1, errors='ignore')
arrivals_df = arrivals_df.reset_index(drop=True)
arrivals_df